# 🛡️ Federated Learning for IoT Network Intrusion Detection (FL-IoT-IDS)
## 🚀 Google Colab Multi-Scenario Experiment Runner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NVKQ2022/FedLearning/blob/main/notebooks/colab_experiment_runner.ipynb)

This notebook provides an automated, end-to-end experimental harness to pull the repository from GitHub and run all intrusion detection benchmark scenarios on Google Colab:
* **Scenario E1:** Centralized PyTorch Baseline (Upper Bound Performance)
* **Scenario E2:** Federated Learning IID Baseline (FedAvg)
* **Scenario E4:** Moderate Statistical Heterogeneity (Dirichlet $\alpha = 0.5$)
* **Scenario E5:** Severe Statistical Heterogeneity (Dirichlet $\alpha = 0.1$ with FedProx $\mu$ Regularization)
* **Comparative Evaluation:** Macro-F1, Minority Recall (Web-based & Brute-force attacks), and Normalized Confusion Matrices.

---
### 🛠️ Step 1: Environment Setup & GitHub Repository Synchronization
We check GPU acceleration, clone the repository `https://github.com/NVKQ2022/FedLearning.git`, enter `/content/FedLearning`, and install dependencies.

In [ ]:
# Check hardware acceleration
import torch
print(f'PyTorch Version: {torch.__version__}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Active Device: {device}')
if torch.cuda.is_available():
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')


In [ ]:
# Clone repository from GitHub (or pull latest updates)
import os, sys, subprocess
repo_url = 'https://github.com/NVKQ2022/FedLearning.git'
repo_dir = '/content/FedLearning'

if not os.path.exists(repo_dir):
    print(f'Cloning {repo_url} to {repo_dir}...')
    subprocess.run(['git', 'clone', repo_url, repo_dir], check=True)
else:
    print('Repository already present. Pulling latest updates from origin/main...')
    subprocess.run(['git', '-C', repo_dir, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', repo_dir, 'reset', '--hard', 'origin/main'], check=True)

# Navigate to repository root
os.chdir(repo_dir)

# Add repository root to sys.path to enable clean imports from src
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

# Purge any cached in-memory src modules so fresh code is loaded
for mod in list(sys.modules.keys()):
    if mod.startswith('src'):
        del sys.modules[mod]

print('Current Working Directory:', os.getcwd())


In [ ]:
# Install required dependencies
!pip install -q -r requirements.txt
print('Dependencies installed successfully.')


In [ ]:
# Set global deterministic seed for scientific reproducibility
from src.utils.seed import set_seed

EXPERIMENT_SEED = 42
set_seed(EXPERIMENT_SEED)
print(f'✅ Deterministic random seed set to: {EXPERIMENT_SEED}')


---
### 💾 Step 2: Dataset Acquisition (Google Drive or Synthetic Fallback)
The raw dataset `merged_CICIOT2023_data.csv` is ~991 MB and excluded from Git.
You can either:
1. Mount Google Drive if you uploaded the CSV there (`/content/drive/MyDrive/FedLearning/...`).
2. Or allow the automatic fallback generator below to synthesize a realistic CICIoT2023-compatible dataset for instant execution.

In [ ]:
# 1. Mount Google Drive to access real dataset
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted successfully.')
except Exception as e:
    print('Drive mount skipped or not running on Colab:', e)

import os, shutil
import pandas as pd, numpy as np

target_csv_path = '/content/FedLearning/datasets/CICIOT2023/merged_CICIOT2023_data.csv'
os.makedirs(os.path.dirname(target_csv_path), exist_ok=True)

# Primary Google Drive path configured for thesis project
gdrive_csv = '/content/drive/MyDrive/NguyenVietKyQuanKLTN/datasets/merged_CICIOT2023_data.csv'

# Fallback check if stored under alternative Drive folder
if not os.path.exists(gdrive_csv):
    alt_gdrive_csv = '/content/drive/MyDrive/FedLearning/merged_CICIOT2023_data.csv'
    if os.path.exists(alt_gdrive_csv):
        gdrive_csv = alt_gdrive_csv

# 2. Check if real dataset exists in Google Drive
if os.path.exists(gdrive_csv):
    gdrive_size = os.path.getsize(gdrive_csv) / (1024**2)
    # Overwrite if target is missing or is the small synthetic dry-run (< 100 MB)
    if not os.path.exists(target_csv_path) or os.path.getsize(target_csv_path) < 100 * 1024 * 1024:
        print(f'Found REAL dataset in Google Drive ({gdrive_size:.2f} MB)! Copying to local Colab SSD...')
        shutil.copy(gdrive_csv, target_csv_path)
        print(f'✅ Real dataset copied successfully: {os.path.getsize(target_csv_path) / (1024**2):.2f} MB')
    else:
        print(f'✅ Real dataset already present on local SSD: {os.path.getsize(target_csv_path) / (1024**2):.2f} MB')
elif not os.path.exists(target_csv_path):
    print('⚠️ Real dataset NOT found at', gdrive_csv)
    print('Please verify that merged_CICIOT2023_data.csv exists under NguyenVietKyQuanKLTN/datasets/ in Google Drive!')
    print('Generating synthetic CICIoT2023-compatible dataset (21 MB) for quick pipeline dry-run...')
    n_samples = 30000
    classes = ['Benign', 'DDoS', 'DoS', 'Recon', 'Spoof', 'Mirai', 'Web-based', 'Brute-force']
    p = [0.18, 0.38, 0.23, 0.10, 0.05, 0.04, 0.015, 0.005]
    
    mock_data = {f'feat_{i}': np.random.exponential(scale=1.5, size=n_samples) for i in range(39)}
    mock_data['feat_0'] = np.random.pareto(a=1.5, size=n_samples) * 10.0
    mock_data['group_class_name'] = np.random.choice(classes, size=n_samples, p=p)
    
    pd.DataFrame(mock_data).to_csv(target_csv_path, index=False)
    print(f'⚠️ Synthetic dataset created ({os.path.getsize(target_csv_path)/(1024**2):.2f} MB). Note: Features are random noise!')
else:
    current_size = os.path.getsize(target_csv_path) / (1024**2)
    if current_size < 100:
        print(f'⚠️ Warning: Currently using SYNTHETIC dry-run dataset ({current_size:.2f} MB, random noise).')
        print('   Upload real merged_CICIOT2023_data.csv (~991 MB) to Google Drive and re-run this cell to train with real data!')
    else:
        print(f'✅ Real CICIoT2023 dataset ready: {current_size:.2f} MB')


---
### 🔄 Step 3: Leak-Free Preprocessing & Class Weights
We use `TabularDataPreprocessor` to clean infinities, impute medians, apply $\log(1+x)$ compression, scale with `RobustScaler`, and compute inverse-frequency class weights ($w_c$).

In [ ]:
from src.data.preprocess import load_and_preprocess_ciciot2023, compute_balanced_class_weights

# Load and preprocess with stratified holdout test split (70/10/20)
data = load_and_preprocess_ciciot2023(
    csv_path=target_csv_path,
    test_size=0.2,
    val_size=0.1,
    sample_size=50000,  # Rapid Colab execution (set None for full dataset)
    scaler_type='robust',
    batch_size=128,
    random_state=EXPERIMENT_SEED
)

class_weights = compute_balanced_class_weights(data['y_train'], num_classes=8)
class_names = data['class_names']
minority_classes = ['Web-based', 'Brute-force']

print(f'Train samples: {len(data["X_train"]):,}')
print(f'Train loader batches: {len(data["train_loader"]):,}')
print(f'Validation samples: {len(data["X_val"]):,}')
print(f'Holdout Test samples: {len(data["X_test"]):,}')
print('Class Names:', class_names)
print('Computed Class Weights:', np.round(class_weights, 3))


---
### 🏆 Step 4: Scenario E1 - Centralized Baseline (Upper Bound)
We train a centralized `TabularIoTMLP` with `MultiClassFocalLoss` and early stopping on validation loss.

In [ ]:
from src.models.mlp import TabularIoTMLP
from src.losses.focal_loss import build_loss_function
from src.optimizers.optimizer import build_optimizer
from src.training.trainer import CentralizedTrainer
from src.evaluation.evaluator import evaluate_comprehensive

# 1. Instantiate model, loss, optimizer
e1_model = TabularIoTMLP(input_dim=39, hidden_dims=(128, 64), num_classes=8, dropout_rate=0.2)
criterion = build_loss_function('focal_loss', class_weights=class_weights, gamma=2.0, device=device)
optimizer = build_optimizer(e1_model, optimizer_type='adamw', lr=1e-3, weight_decay=1e-4)

# 2. Fit with CentralizedTrainer
trainer_e1 = CentralizedTrainer(e1_model, optimizer, criterion, device=device, max_grad_norm=5.0)
history_e1 = trainer_e1.fit(data['train_loader'], data['val_loader'], epochs=10, patience=3, verbose=True)

# 3. Comprehensive holdout evaluation
e1_results = evaluate_comprehensive(
    model=e1_model,
    dataloader=data['test_loader'],
    class_names=class_names,
    minority_classes=minority_classes,
    device=device
)

print(f'Scenario E1 Centralized Test Macro-F1: {e1_results["macro_f1"]*100:.2f}%')
print(f'Overall Accuracy: {e1_results["accuracy"]*100:.2f}%')
print('Minority Recall:', e1_results['minority_recall'])


---
### 🌐 Step 5: Scenario E2 - Federated IID Baseline (FedAvg)
Data is partitioned uniformly across $K=5$ edge clients using Stratified IID partitioning.

In [ ]:
from src.data.partition import partition_iid, create_client_dataloaders
from src.training.trainer import LocalClientTrainer
import copy

num_clients = 5
num_rounds = 8
local_epochs = 2

iid_parts = partition_iid(data['y_train'], num_clients=num_clients, seed=EXPERIMENT_SEED)
client_loaders_iid = create_client_dataloaders(data['X_train'], data['y_train'], iid_parts, batch_size=64)

global_model_e2 = TabularIoTMLP(input_dim=39, num_classes=8).to(device)

print(f'Starting Scenario E2: {num_clients} clients, {num_rounds} rounds, {local_epochs} local epochs...')
for round_idx in range(1, num_rounds + 1):
    client_weights = []
    for client_id in range(num_clients):
        local_m = copy.deepcopy(global_model_e2)
        opt = build_optimizer(local_m, 'adamw', lr=1e-3)
        c_trainer = LocalClientTrainer(local_m, opt, criterion, device=device)
        c_trainer.train_epochs(client_loaders_iid[client_id], num_epochs=local_epochs, mu=0.0)
        client_weights.append(local_m.get_weights())
    
    # FedAvg Server Aggregation
    avg_w = [np.mean([cw[l] for cw in client_weights], axis=0) for l in range(len(client_weights[0]))]
    global_model_e2.set_weights(avg_w)
    print(f'  [Round {round_idx:02d}/{num_rounds:02d}] Aggregation complete.')

e2_results = evaluate_comprehensive(
    model=global_model_e2,
    dataloader=data['test_loader'],
    class_names=class_names,
    minority_classes=minority_classes,
    device=device
)
print(f'Scenario E2 IID FedAvg Test Macro-F1: {e2_results["macro_f1"]*100:.2f}%')
print('Minority Recall:', e2_results['minority_recall'])


---
### 🔥 Step 6: Scenario E5 - Severe Non-IID Dirichlet Skew (FedAvg vs. FedProx)
We simulate severe label heterogeneity ($\alpha = 0.1$) where rare attacks are concentrated in single edge clients.
We evaluate **FedProx** ($\mu = 0.05$) to constrain client parameter drift.

In [ ]:
from src.data.partition import partition_dirichlet

# 1. Generate Dirichlet Non-IID partition (alpha = 0.1)
non_iid_parts = partition_dirichlet(
    y=data['y_train'],
    num_clients=num_clients,
    alpha=0.1,
    min_samples_per_client=100,
    seed=EXPERIMENT_SEED
)
client_loaders_non_iid = create_client_dataloaders(data['X_train'], data['y_train'], non_iid_parts, batch_size=64)

# 2. Train with FedProx (mu = 0.05)
global_model_e5 = TabularIoTMLP(input_dim=39, num_classes=8).to(device)
mu_val = 0.05

print(f'Starting Scenario E5 (Dirichlet alpha=0.1, FedProx mu={mu_val})...')
for round_idx in range(1, num_rounds + 1):
    client_weights = []
    for client_id in range(num_clients):
        local_m = copy.deepcopy(global_model_e5)
        opt = build_optimizer(local_m, 'adamw', lr=1e-3)
        c_trainer = LocalClientTrainer(local_m, opt, criterion, device=device)
        
        # FedProx proximal penalty: (mu / 2) * ||w - w_t||^2
        c_trainer.train_epochs(
            client_loaders_non_iid[client_id],
            num_epochs=local_epochs,
            global_model=global_model_e5,
            mu=mu_val
        )
        client_weights.append(local_m.get_weights())
    
    avg_w = [np.mean([cw[l] for cw in client_weights], axis=0) for l in range(len(client_weights[0]))]
    global_model_e5.set_weights(avg_w)
    print(f'  [Round {round_idx:02d}/{num_rounds:02d}] Aggregation complete.')

e5_results = evaluate_comprehensive(
    model=global_model_e5,
    dataloader=data['test_loader'],
    class_names=class_names,
    minority_classes=minority_classes,
    device=device
)
print(f'Scenario E5 Non-IID FedProx Test Macro-F1: {e5_results["macro_f1"]*100:.2f}%')
print('Minority Recall:', e5_results['minority_recall'])


---
### 📊 Step 7: Comparative Analysis & Publication Figures
We plot comparative performance bars and the normalized confusion matrix across scenarios.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. Macro-F1 Comparison Bar Chart
scenarios = ['E1: Centralized\n(Upper Bound)', 'E2: Federated IID\n(FedAvg)', 'E5: Severe Non-IID\n(FedProx $\\mu=0.05$)']
macro_scores = [e1_results['macro_f1'] * 100, e2_results['macro_f1'] * 100, e5_results['macro_f1'] * 100]
colors = ['#2ecc71', '#3498db', '#e74c3c']

bars = axes[0].bar(scenarios, macro_scores, color=colors, edgecolor='black', width=0.5)
axes[0].set_ylabel('Macro-F1 Score (%)', fontsize=12)
axes[0].set_ylim(0, 105)
axes[0].set_title('Intrusion Detection Performance Across Scenarios', fontsize=13, fontweight='bold')
for bar in bars:
    yval = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2.0, yval + 1.5, f'{yval:.2f}%', ha='center', va='bottom', fontweight='bold')

# 2. Normalized Confusion Matrix for Scenario E5
cm = e5_results['confusion_matrix']
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[1], cbar=False)
axes[1].set_title('Scenario E5 Normalized Confusion Matrix', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predicted Class', fontsize=12)
axes[1].set_ylabel('True Class', fontsize=12)
plt.setp(axes[1].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
os.makedirs('reports/figures', exist_ok=True)
plt.savefig('reports/figures/colab_scenario_comparison.png', dpi=300, bbox_inches='tight')
plt.show()


---
### 💾 Step 8: Persist Results & Metrics to Google Drive
Export metrics and figures to Google Drive to protect against Colab runtime disconnections.

In [ ]:
import json

export_dict = {
    'E1_Centralized': {k: v for k, v in e1_results.items() if k != 'confusion_matrix'},
    'E2_IID_FedAvg': {k: v for k, v in e2_results.items() if k != 'confusion_matrix'},
    'E5_Severe_Non_IID_FedProx': {k: v for k, v in e5_results.items() if k != 'confusion_matrix'},
}

local_metrics_file = 'reports/colab_benchmark_metrics.json'
with open(local_metrics_file, 'w') as f:
    json.dump(export_dict, f, indent=4)
print(f'Metrics saved locally to {local_metrics_file}')

# Optional export to Google Drive if mounted
drive_reports_dir = '/content/drive/MyDrive/NguyenVietKyQuanKLTN/reports'
if not os.path.exists('/content/drive/MyDrive/NguyenVietKyQuanKLTN') and os.path.exists('/content/drive/MyDrive/FedLearning'):
    drive_reports_dir = '/content/drive/MyDrive/FedLearning/reports'
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(drive_reports_dir, exist_ok=True)
    shutil.copy(local_metrics_file, os.path.join(drive_reports_dir, 'colab_benchmark_metrics.json'))
    shutil.copy('reports/figures/colab_scenario_comparison.png', os.path.join(drive_reports_dir, 'colab_scenario_comparison.png'))
    print(f'Results and plots successfully archived to Google Drive: {drive_reports_dir}')
else:
    print('Google Drive not mounted. Results stored locally in Colab runtime.')
